In [ ]:
!pip install -q "transformers>=4.40,<5.0" accelerate torchaudio librosa scikit-learn
!apt-get -y -q install ffmpeg > /dev/null 2>&1

!git clone -q https://github.com/m3hrdadfi/soxan.git
import sys
sys.path.append("/content/soxan")
from src.models import Wav2Vec2ForSpeechClassification

import torch
print("GPU:", torch.cuda.is_available())

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 71.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 26.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.
GPU: True


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from transformers import AutoConfig, Wav2Vec2FeatureExtractor

PHASE1_CHECKPOINT = "/content/drive/MyDrive/final_project/shemo_phase1_checkpoint"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(PHASE1_CHECKPOINT)
target_sampling_rate = feature_extractor.sampling_rate

frozen_model = Wav2Vec2ForSpeechClassification.from_pretrained(PHASE1_CHECKPOINT).to(device)
frozen_model.eval()

print("Model loaded. Sampling rate:", target_sampling_rate)

Model loaded. Sampling rate: 16000


In [ ]:
import glob, os, subprocess
from collections import Counter

REAL_DATA_DIR = "/content/drive/MyDrive/final_project/voice_dataset/audio"
CONVERTED_DIR = "/content/real_audio_wav"

# Find all .m4a files
m4a_files = glob.glob(f"{REAL_DATA_DIR}/**/*.m4a", recursive=True)
print(f"Number of .m4a files: {len(m4a_files)}")

# Convert .m4a files to .wav (16kHz, mono)
os.makedirs(CONVERTED_DIR, exist_ok=True)
for f in m4a_files:
    speaker = os.path.basename(os.path.dirname(f))
    out_dir = os.path.join(CONVERTED_DIR, speaker)
    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, os.path.splitext(os.path.basename(f))[0] + ".wav")
    if not os.path.exists(out_path):
        subprocess.run(["ffmpeg", "-y", "-i", f, "-ar", "16000", "-ac", "1", out_path],
                        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Map emotion codes to standardized labels
LABEL_CODE_MAP = {"ANG": "anger", "HAP": "happiness", "NEU": "neutral", "SAD": "sadness"}

def get_label_from_filename(fp):
    name = os.path.splitext(os.path.basename(fp))[0].upper()
    for code, emo in LABEL_CODE_MAP.items():
        if code in name:
            return emo
    return None

# Load real data with labels and speaker IDs
speaker_dirs = sorted(glob.glob(f"{CONVERTED_DIR}/speaker_*"), key=lambda p: int(p.split("_")[-1]))
real_data = []  # (path, label, speaker_id)
for d in speaker_dirs:
    sid = os.path.basename(d)
    for f in glob.glob(f"{d}/*.wav"):
        lb = get_label_from_filename(f)
        if lb:
            real_data.append((f, lb, sid))

print(f"Number of speakers: {len(speaker_dirs)} | Number of samples: {len(real_data)}")
print("Class distribution:", Counter([l for _, l, _ in real_data]))

Number of .m4a files: 112
Number of speakers: 26 | Number of samples: 112
Class distribution: Counter({'happiness': 30, 'sadness': 28, 'anger': 27, 'neutral': 27})


In [ ]:
import torchaudio

def speech_file_to_array_fn(path, target_sr):
    speech_array, orig_sr = torchaudio.load(path)
    if speech_array.shape[0] > 1:
        speech_array = speech_array.mean(dim=0, keepdim=True)
    speech = torchaudio.transforms.Resample(orig_sr, target_sr)(speech_array).squeeze().numpy()
    return speech

In [ ]:
import librosa
import numpy as np

def extract_prosodic_features(speech, sr):
    f0, _, _ = librosa.pyin(speech, fmin=librosa.note_to_hz('C2'), fmax=librosa.note_to_hz('C7'), sr=sr)
    f0_voiced = f0[~np.isnan(f0)]
    if len(f0_voiced) == 0:
        f0_mean, f0_std, f0_range = 0.0, 0.0, 0.0
    else:
        f0_mean, f0_std = float(np.mean(f0_voiced)), float(np.std(f0_voiced))
        f0_range = float(np.max(f0_voiced) - np.min(f0_voiced))

    rms = librosa.feature.rms(y=speech)[0]
    rms_mean, rms_std = float(np.mean(rms)), float(np.std(rms))

    zcr_mean = float(np.mean(librosa.feature.zero_crossing_rate(speech)[0]))

    centroid = librosa.feature.spectral_centroid(y=speech, sr=sr)[0]
    centroid_mean, centroid_std = float(np.mean(centroid)), float(np.std(centroid))

    return np.array([f0_mean, f0_std, f0_range, rms_mean, rms_std, zcr_mean, centroid_mean, centroid_std])

In [ ]:
@torch.no_grad()
def extract_wav2vec_embedding(speech):
    """Extract Wav2Vec2 embedding from speech audio"""
    inputs = feature_extractor(speech, sampling_rate=target_sampling_rate, return_tensors="pt", padding=True)
    input_values = inputs["input_values"].to(device)
    hidden_states = frozen_model.wav2vec2(input_values).last_hidden_state
    return hidden_states.mean(dim=1).squeeze(0).cpu().numpy()

# Extract combined features (Wav2Vec2 + prosodic) for all real data
combined_features, labels, speakers = [], [], []

for i, (path, label, speaker_id) in enumerate(real_data):
    speech = speech_file_to_array_fn(path, target_sampling_rate)
    wav2vec_emb = extract_wav2vec_embedding(speech)  # 1024-dim embedding
    prosody = extract_prosodic_features(speech, target_sampling_rate)  # 8 prosodic features
    combined_features.append(np.concatenate([wav2vec_emb, prosody]))  # 1032-dim combined
    labels.append(label)
    speakers.append(speaker_id)
    if (i + 1) % 20 == 0:
        print(f"{i+1}/{len(real_data)} processed")

combined_features = np.stack(combined_features)
print("Final feature shape (1024 + 8):", combined_features.shape)

20/112 processed
40/112 processed
60/112 processed
80/112 processed
100/112 processed
Final feature shape (1024 + 8): (112, 1032)


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
import pandas as pd

# Leave-one-speaker-out cross-validation
unique_speakers = sorted(set(speakers), key=lambda s: int(s.split("_")[-1]))
all_true, all_pred = [], []

for test_speaker in unique_speakers:
    # Split data by speaker
    train_idx = [i for i, s in enumerate(speakers) if s != test_speaker]
    test_idx = [i for i, s in enumerate(speakers) if s == test_speaker]
    if not test_idx:
        continue

    X_train, y_train = combined_features[train_idx], [labels[i] for i in train_idx]
    X_test, y_test = combined_features[test_idx], [labels[i] for i in test_idx]

    # Standardize features
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)

    # Train logistic regression classifier
    clf = LogisticRegression(max_iter=2000, class_weight="balanced", C=1.0)
    clf.fit(X_train_s, y_train)
    preds = clf.predict(X_test_s)

    # Store results
    all_true.extend(y_test)
    all_pred.extend(preds)

# Print evaluation metrics
print(f"Accuracy (embedding + prosody): {accuracy_score(all_true, all_pred):.4f}")
print(f"F1 macro: {f1_score(all_true, all_pred, average='macro'):.4f}")
print(classification_report(all_true, all_pred))

# Display confusion matrix
labels_order = sorted(set(all_true) | set(all_pred))
cm = confusion_matrix(all_true, all_pred, labels=labels_order)
pd.DataFrame(cm, index=labels_order, columns=labels_order)

Accuracy (embedding + prosody): 0.6964
F1 macro: 0.6944
              precision    recall  f1-score   support

       anger       0.67      0.59      0.63        27
   happiness       0.79      0.63      0.70        30
     neutral       0.63      0.81      0.71        27
     sadness       0.72      0.75      0.74        28

    accuracy                           0.70       112
   macro avg       0.70      0.70      0.69       112
weighted avg       0.71      0.70      0.70       112



,anger,happiness,neutral,sadness
anger,16,3,4,4
happiness,4,19,6,1
neutral,0,2,22,3
sadness,4,0,3,21


In [ ]:
# Extract only prosodic features (last 8 dimensions)
prosody_only = combined_features[:, -8:]

# Evaluate using only prosodic features (baseline comparison)
all_true_p, all_pred_p = [], []
for test_speaker in unique_speakers:
    train_idx = [i for i, s in enumerate(speakers) if s != test_speaker]
    test_idx = [i for i, s in enumerate(speakers) if s == test_speaker]
    if not test_idx:
        continue
    X_train, y_train = prosody_only[train_idx], [labels[i] for i in train_idx]
    X_test, y_test = prosody_only[test_idx], [labels[i] for i in test_idx]
    scaler = StandardScaler()
    clf = LogisticRegression(max_iter=2000, class_weight="balanced")
    clf.fit(scaler.fit_transform(X_train), y_train)
    all_true_p.extend(y_test)
    all_pred_p.extend(clf.predict(scaler.transform(X_test)))

print(f"Accuracy with prosody only: {accuracy_score(all_true_p, all_pred_p):.4f}")

Accuracy with prosody only: 0.3750


In [ ]:
import joblib
import json

# Train final model on all 112 samples (combined_features and labels are already in memory)
final_scaler = StandardScaler()
X_all_s = final_scaler.fit_transform(combined_features)

final_clf = LogisticRegression(max_iter=2000, class_weight="balanced", C=1.0)
final_clf.fit(X_all_s, labels)

# Save the final model and scaler
FINAL_DIR = "/content/drive/MyDrive/final_project/final_classifier_v2"
os.makedirs(FINAL_DIR, exist_ok=True)

joblib.dump(final_clf, f"{FINAL_DIR}/logistic_classifier.joblib")
joblib.dump(final_scaler, f"{FINAL_DIR}/scaler.joblib")

# Metadata to know exactly what this model depends on (for the inference pipeline later)
metadata = {
    "phase1_checkpoint": PHASE1_CHECKPOINT,
    "feature_order": (
        ["wav2vec2_embedding_dim_" + str(i) for i in range(1024)] +
        ["f0_mean", "f0_std", "f0_range", "rms_mean", "rms_std", "zcr_mean", "centroid_mean", "centroid_std"]
    ),
    "total_feature_dim": combined_features.shape[1],
    "classes": list(final_clf.classes_),
    "loso_accuracy": 0.6964,
    "loso_f1_macro": 0.6944,
    "n_training_samples": len(labels),
}
with open(f"{FINAL_DIR}/metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print(f"Final model saved to: {FINAL_DIR}")
print("Saved files: logistic_classifier.joblib, scaler.joblib, metadata.json")
print("Classes:", final_clf.classes_)

Final model saved to: /content/drive/MyDrive/final_project/final_classifier_v2
Saved files: logistic_classifier.joblib, scaler.joblib, metadata.json
Classes: ['anger' 'happiness' 'neutral' 'sadness']
